# Multisite Transport Memory Classification

This notebook performs comparative analysis of recovered memory functions from multiple transport sites.

Methods:

1. Wasserstein distance analysis
2. Dynamic Time Warping (DTW) similarity
3. Functional Principal Component Analysis (FPCA)
4. Memory-function clustering

The goal is to identify transferable transport memory classes between sites.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parents[0]
sys.path.append(str(ROOT))


## Import multisite analysis modules


In [ ]:
from src.multisite.wasserstein import wasserstein_distance_matrix
from src.multisite.dtw import dtw_distance_matrix
from src.multisite.fpca import perform_fpca
from src.multisite.clustering import cluster_memory_functions


## Generate example memory functions from multiple sites

Each row represents one site memory function H(t).


In [ ]:
time = np.linspace(0,100,300)

site_01 = np.exp(-0.05*time)
site_02 = np.exp(-0.025*time)
site_03 = (time+1)**(-0.6)
site_04 = 0.5*np.exp(-0.04*time)+0.5*(time+1)**(-0.4)

memory_functions = np.vstack([
    site_01,
    site_02,
    site_03,
    site_04
])

plt.figure(figsize=(8,5))

for i,H in enumerate(memory_functions):
    plt.plot(time,H,label=f'Site {i+1}')

plt.xlabel('Time')
plt.ylabel('H(t)')
plt.title('Memory Functions Across Sites')
plt.legend()
plt.show()


## Wasserstein distance between sites

The Wasserstein metric measures the difference between memory distributions.

In [ ]:
W = wasserstein_distance_matrix(
    time,
    memory_functions
)

print(W)

plt.figure(figsize=(6,5))
plt.imshow(W)
plt.colorbar(label='Wasserstein distance')
plt.xlabel('Site')
plt.ylabel('Site')
plt.title('Wasserstein Similarity Matrix')
plt.show()


## Dynamic Time Warping analysis

DTW identifies similarity between memory shapes with different characteristic times.

In [ ]:
DTW = dtw_distance_matrix(
    memory_functions
)

print(DTW)


## Functional Principal Component Analysis

Memory functions are represented using dominant variability modes.

In [ ]:
fpca_result = perform_fpca(
    memory_functions,
    n_components=2
)

scores = fpca_result['scores']

print('FPCA scores:')
print(scores)

plt.figure(figsize=(6,5))
plt.scatter(scores[:,0], scores[:,1])

for i,(x,y) in enumerate(scores):
    plt.text(x,y,f'Site {i+1}')

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('FPCA Memory Space')
plt.show()


## Cluster transport memory classes


In [ ]:
clusters = cluster_memory_functions(
    memory_functions,
    n_clusters=2
)

labels = clusters['labels']

for i,label in enumerate(labels):
    print(f'Site {i+1}: Cluster {label}')


## Interpretation

Clusters represent groups of sites with similar transport memory behavior.

Possible interpretations:

- exchange-dominated transport
- diffusion-controlled long-memory transport
- mixed transport regimes
